In [ ]:
# ============================================================
# 🏓 TTMATCH — 強化版 Training & Submission（可直接執行）
# ============================================================

# 若在本機/Colab 沒安裝套件（Kaggle 通常內建）
# !pip install -q scikit-learn lightgbm

import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import GroupKFold, KFold, train_test_split
from sklearn.metrics import f1_score, roc_auc_score, classification_report
from lightgbm import LGBMClassifier

# ---------- 0) 路徑 ----------
CLEAN_TRAIN = "data/clean_train.csv"
CLEAN_TEST  = "data/clean_test.csv"
SAMPLE_SUB  = "data/sample_submission.csv"

# ---------- 1) 讀檔 ----------
clean_train = pd.read_csv(CLEAN_TRAIN, low_memory=False)
clean_test  = pd.read_csv(CLEAN_TEST,  low_memory=False)
sample_sub  = pd.read_csv(SAMPLE_SUB,  low_memory=False)

print("clean_train:", clean_train.shape)
print("clean_test :", clean_test.shape)
print("sample_sub :", sample_sub.shape)

# ---------- 2) 基本設定 ----------
KEY = "rally_uid"
assert KEY in clean_test.columns and KEY in sample_sub.columns, f"❌ 找不到對齊鍵 {KEY}"

ACTION_COL = "next_actionId"
POINT_COL  = "next_pointId"
assert ACTION_COL in clean_train.columns, f"❌ 缺少 {ACTION_COL}"
assert POINT_COL  in clean_train.columns, f"❌ 缺少 {POINT_COL}"

RALLY_TARGET_CANDIDATES = ["serverGetPoint", "server_won_point"]
RALLY_TARGET = next((c for c in RALLY_TARGET_CANDIDATES if c in clean_train.columns), None)
print(f"✅ 標籤：{ACTION_COL}, {POINT_COL}, 任務3：{RALLY_TARGET}")

GROUP_COL = "match_id" if "match_id" in clean_train.columns else ("rally_id" if "rally_id" in clean_train.columns else None)
groups = clean_train[GROUP_COL] if GROUP_COL and GROUP_COL in clean_train.columns else None
if groups is None:
    print("⚠️ 找不到 match_id/rally_id，CV 將退化為隨機切分。")

# ---------- 3) 進階特徵工程（K=5 + pair + bigram + phase + is_self_server） ----------
def phase_from_stroke(s):
    # 1=發球, 2=接發, 4=第三板之後，其餘=3
    return np.where(s==1, 0, np.where(s==2, 1, np.where(s==4, 2, 3)))

for df in (clean_train, clean_test):
    if df is None: continue
    # phase
    if "stroke_id" in df.columns:
        df["phase"] = phase_from_stroke(df["stroke_id"])
    # is_self_server
    if {"server_id","player_self_id"}.issubset(df.columns):
        df["is_self_server"] = (df["server_id"] == df["player_self_id"]).astype(int)
    # prev1_pair
    if {"prev1_actionId","prev1_pointId"}.issubset(df.columns):
        df["prev1_pair"] = (df["prev1_actionId"].astype("Int64").astype(str)
                            + "_" + df["prev1_pointId"].astype("Int64").astype(str))
        df["prev1_pair"] = df["prev1_pair"].astype("category").cat.codes
    # prev2→prev1 bigram
    if {"prev1_actionId","prev2_actionId"}.issubset(df.columns):
        df["prev2to1_action_bigram"] = (df["prev2_actionId"].astype("Int64").astype(str)
                                        + ">" + df["prev1_actionId"].astype("Int64").astype(str))
        df["prev2to1_action_bigram"] = df["prev2to1_action_bigram"].astype("category").cat.codes

# ---------- 4) 特徵清單 ----------
base_feats = [
    "prev1_actionId","prev2_actionId","prev3_actionId","prev4_actionId","prev5_actionId",
    "prev1_pointId","prev2_pointId","prev3_pointId","prev4_pointId","prev5_pointId",
    "prev1_pair","prev2to1_action_bigram","phase","is_self_server",
    "score_diff","rally_progress","rally_length",
    "sex","gamePlayerId","gamePlayerOtherId","player_self_id","player_opponent_id",
    "position_id","spin_id","strength_id","handId","stroke_id","action_id","point_id"
]
delta_feats = [c for c in clean_train.columns if c.startswith("delta_")]
features_all = [c for c in base_feats + delta_feats if c in clean_train.columns]
print("實際使用特徵數：", len(features_all))

# ---------- 5) 訓練集補值（prev*/類別→-1；連續→0） ----------
train_df = clean_train.copy()
prev_cols_tr = [c for c in train_df.columns if c.startswith(("prev1_","prev2_","prev3_","prev4_","prev5_"))]
maybe_cat_tr = ["sex","gamePlayerId","gamePlayerOtherId","player_self_id","player_opponent_id",
                "position_id","spin_id","strength_id","handId","stroke_id","action_id","point_id",
                "prev1_pair","prev2to1_action_bigram","phase","is_self_server"]
cat_cols_tr = [c for c in maybe_cat_tr if c in train_df.columns]
num_cols_tr = [c for c in ["score_diff","rally_progress","rally_length"] if c in train_df.columns]

if prev_cols_tr: train_df[prev_cols_tr] = train_df[prev_cols_tr].fillna(-1)
if cat_cols_tr:  train_df[cat_cols_tr]  = train_df[cat_cols_tr].fillna(-1)
if num_cols_tr:  train_df[num_cols_tr]  = train_df[num_cols_tr].fillna(0.0)
train_df = train_df.fillna(0)

X_all = train_df[features_all]
y_action = train_df[ACTION_COL].astype(int)
y_point  = train_df[POINT_COL].astype(int)

# ---------- 6) OOF Target Encoding（不洩漏） ----------
def oof_target_encode(df, feat_cols, target_col, n_splits=5, smoothing=100, global_prior=None, seed=42):
    """
    多分類 OOF target encoding：對 feat_cols -> target_col 產生每類別機率。
    回傳：新 df、新欄位名、類別順序、全域 prior
    """
    df = df.copy()
    y = df[target_col].astype(int).values
    classes = np.unique(y)
    if global_prior is None:
        prior_s = df[target_col].value_counts(normalize=True).sort_index()
        prior_s = prior_s.reindex(classes, fill_value=0)
        global_prior = prior_s.values

    new_cols = [f"te_{'_'.join(feat_cols)}__{target_col}__c{c}" for c in classes]
    oof = np.zeros((len(df), len(classes)), dtype=float)

    kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
    for tr_idx, va_idx in kf.split(df, y):
        tr, va = df.iloc[tr_idx], df.iloc[va_idx]
        key = tr.groupby(feat_cols)[target_col].value_counts().unstack(fill_value=0)
        key = key.reindex(columns=classes, fill_value=0)
        cnt = key.sum(axis=1).values.reshape(-1,1)
        prob = (key.values + (smoothing * global_prior)) / (cnt + smoothing)  # smoothing

        # mapping
        join = va[feat_cols].merge(key.reset_index(), on=feat_cols, how="left").drop(columns=feat_cols)
        if join.shape[1] == 0:
            va_prob = np.tile(global_prior, (len(va),1))
        else:
            join = join.reindex(columns=classes, fill_value=0)
            arr  = join.values
            cnt2 = arr.sum(axis=1, keepdims=True)
            va_prob = (arr + (smoothing * global_prior)) / (cnt2 + smoothing)
        oof[va_idx] = va_prob

    for i, col in enumerate(new_cols):
        df[col] = oof[:, i]
    return df, new_cols, classes, global_prior

# 定義情境組合（你可再加/改）
fe_sets = []
if "prev1_actionId" in train_df.columns: fe_sets.append(["prev1_actionId"])
if {"prev1_actionId","phase"}.issubset(train_df.columns): fe_sets.append(["prev1_actionId","phase"])
if {"prev1_actionId","prev1_pointId"}.issubset(train_df.columns): fe_sets.append(["prev1_actionId","prev1_pointId"])
if {"prev1_actionId","is_self_server"}.issubset(train_df.columns): fe_sets.append(["prev1_actionId","is_self_server"])

train_te = train_df.copy()
te_cols_act = []
for feats in fe_sets:
    train_te, cols, action_classes, action_prior = oof_target_encode(
        train_te, feats, ACTION_COL, n_splits=5, smoothing=120, seed=42
    )
    te_cols_act += cols

train_te2 = train_df.copy()
te_cols_point = []
for feats in fe_sets:
    train_te2, cols, point_classes, point_prior = oof_target_encode(
        train_te2, feats, POINT_COL, n_splits=5, smoothing=120, seed=42
    )
    te_cols_point += cols

X_all_te       = pd.concat([X_all,       train_te[te_cols_act]],   axis=1)
X_all_te_point = pd.concat([X_all,       train_te2[te_cols_point]], axis=1)

# ---------- 7) CV 函式 ----------
def cv_macro_f1(X, y, groups=None, n_splits=5, random_state=42):
    scores = []
    if groups is not None:
        gkf = GroupKFold(n_splits=n_splits)
        split_iter = gkf.split(X, y, groups)
    else:
        X_tr, X_va, y_tr, y_va = train_test_split(X, y, test_size=0.2, random_state=random_state, stratify=y)
        split_iter = [ (X_tr.index, X_va.index) ]
    for tr_idx, va_idx in split_iter:
        model = LGBMClassifier(
            n_estimators=1200, learning_rate=0.03, num_leaves=255,
            max_depth=-1, min_child_samples=20, subsample=0.9, colsample_bytree=0.9,
            reg_alpha=1.0, reg_lambda=2.0,
            random_state=random_state, n_jobs=-1, class_weight="balanced"
        )
        model.fit(X.iloc[tr_idx], y.iloc[tr_idx])
        pred = model.predict(X.iloc[va_idx])
        scores.append(f1_score(y.iloc[va_idx], pred, average="macro"))
    return float(np.mean(scores))

def cv_auc(X, y, groups=None, n_splits=5, random_state=42):
    scores = []
    if groups is not None:
        gkf = GroupKFold(n_splits=n_splits)
        split_iter = gkf.split(X, y, groups)
    else:
        X_tr, X_va, y_tr, y_va = train_test_split(X, y, test_size=0.2, random_state=random_state, stratify=y)
        split_iter = [ (X_tr.index, X_va.index) ]
    for tr_idx, va_idx in split_iter:
        model = LGBMClassifier(
            n_estimators=1200, learning_rate=0.03, num_leaves=255,
            max_depth=-1, min_child_samples=20, subsample=0.9, colsample_bytree=0.9,
            reg_alpha=1.0, reg_lambda=2.0,
            random_state=random_state, n_jobs=-1, class_weight="balanced"
        )
        model.fit(X.iloc[tr_idx], y.iloc[tr_idx])
        prob = model.predict_proba(X.iloc[va_idx])[:, 1]
        scores.append(roc_auc_score(y.iloc[va_idx], prob))
    return float(np.mean(scores))

# ---------- 8) 任務1/2 CV + 全量訓練（加權 & 調參） ----------
print("CV Task1 (action) ...")
f1_action_cv = cv_macro_f1(X_all_te, y_action, groups)
print(f"Task1 Macro F1 (CV): {f1_action_cv:.4f}")

print("CV Task2 (point) ...")
f1_point_cv = cv_macro_f1(X_all_te_point, y_point, groups)
print(f"Task2 Macro F1 (CV): {f1_point_cv:.4f}")

lgb_params_mc = dict(
    n_estimators=1400, learning_rate=0.03, num_leaves=255,
    max_depth=-1, min_child_samples=20, subsample=0.9, colsample_bytree=0.9,
    reg_alpha=1.0, reg_lambda=2.0,
    random_state=42, n_jobs=-1, class_weight="balanced"
)

model_action_full = LGBMClassifier(**lgb_params_mc)
model_action_full.fit(X_all_te, y_action)

model_point_full  = LGBMClassifier(**lgb_params_mc)
model_point_full.fit(X_all_te_point, y_point)

# ---------- 9) 任務3（AUC） ----------
if RALLY_TARGET is not None:
    remove_prefixes = ("prev1_","prev2_","prev3_","prev4_","prev5_")
    feats_rally = [f for f in features_all if not f.startswith(remove_prefixes)]
    if not feats_rally:
        feats_rally = [c for c in ["score_diff","rally_progress","rally_length"] if c in train_df.columns]
    X_rally = train_df[feats_rally]
    y_rally = train_df[RALLY_TARGET].astype(int)

    auc_rally_cv = cv_auc(X_rally, y_rally, groups)
    print(f"Task3 AUC (CV): {auc_rally_cv:.4f}")

    lgb_params_bin = dict(
        n_estimators=1200, learning_rate=0.03, num_leaves=255,
        max_depth=-1, min_child_samples=20, subsample=0.9, colsample_bytree=0.9,
        reg_alpha=1.0, reg_lambda=2.0,
        random_state=42, n_jobs=-1, class_weight="balanced"
    )
    model_rally_full = LGBMClassifier(**lgb_params_bin)
    model_rally_full.fit(X_rally, y_rally)
else:
    auc_rally_cv = None
    model_rally_full = None
    print("⚠️ 找不到二分類目標（serverGetPoint/server_won_point），跳過 Task3。")

# ---------- 10) 參考 blended 分數 ----------
try:
    blended = 0.4*float(f1_action_cv) + 0.4*float(f1_point_cv) + 0.2*float(auc_rally_cv if auc_rally_cv is not None else 0.5)
    print(f"Offline blended score (ref): {blended:.4f}")
except Exception as e:
    print("無法計算 blended：", e)

# ---------- 11) 對齊 test → 只保留每個 rally_uid 的最後一擊，並依 sample_sub 的順序重排 ----------
test_df = clean_test.copy()
test_df[KEY] = test_df[KEY].astype(str)
sample_sub[KEY] = sample_sub[KEY].astype(str)

if "stroke_number" in test_df.columns:
    dedup = (test_df.sort_values([KEY, "stroke_number"])
                  .groupby(KEY, as_index=False)
                  .tail(1))
else:
    dedup = test_df.drop_duplicates(subset=[KEY], keep="last")

test_sel = (dedup.set_index(KEY)
                 .reindex(sample_sub[KEY])
                 .reset_index())

# ---------- 12) 測試集補值與同樣特徵工程 ----------
prev_cols_te = [c for c in test_sel.columns if c.startswith(("prev1_","prev2_","prev3_","prev4_","prev5_"))]
maybe_cat_te = ["sex","gamePlayerId","gamePlayerOtherId","player_self_id","player_opponent_id",
                "position_id","spin_id","strength_id","handId","stroke_id","action_id","point_id",
                "prev1_pair","prev2to1_action_bigram","phase","is_self_server"]
cat_cols_te = [c for c in maybe_cat_te if c in test_sel.columns]
num_cols_te = [c for c in ["score_diff","rally_progress","rally_length"] if c in test_sel.columns]

if prev_cols_te: test_sel[prev_cols_te] = test_sel[prev_cols_te].fillna(-1)
if cat_cols_te:  test_sel[cat_cols_te]  = test_sel[cat_cols_te].fillna(-1)
if num_cols_te:  test_sel[num_cols_te]  = test_sel[num_cols_te].fillna(0.0)
test_sel = test_sel.fillna(0)

# 先組 base 特徵
features_in_test = [f for f in features_all if f in test_sel.columns]
X_test_base = test_sel[features_in_test].copy()

# ---------- 13) 以訓練集的條件表，將 TE 映射到 test ----------
def build_cond_table(df, feat_cols, target_col):
    tbl = df.groupby(feat_cols)[target_col].value_counts(normalize=True).unstack(fill_value=0)
    tbl = tbl.sort_index(axis=1)
    return tbl

def map_cond_probs_to_test(train_df, test_df, feat_cols, target_col, classes, global_prior, prefix):
    key_cols = feat_cols.copy()
    tbl = build_cond_table(train_df, feat_cols, target_col).reset_index()
    merged = test_df[key_cols].merge(tbl, on=key_cols, how="left").drop(columns=key_cols)
    # 補上缺列/缺欄
    for c in classes:
        colname = c if isinstance(c, str) else int(c)
        if colname not in merged.columns:
            merged[colname] = np.nan
    merged = merged.reindex(columns=classes, fill_value=np.nan)
    arr = merged.values
    mask = np.isnan(arr)
    arr[mask] = np.take(global_prior, np.where(mask)[1])
    new_cols = [f"{prefix}__c{c}" for c in classes]
    out_df = pd.DataFrame(arr, columns=new_cols, index=test_df.index)
    return out_df, new_cols

X_test_te_list = []
for feats in fe_sets:
    exist = [c for c in feats if c in train_df.columns and c in test_sel.columns]
    if not exist: 
        continue
    te_df, te_names = map_cond_probs_to_test(train_df, test_sel, exist, ACTION_COL, action_classes, action_prior,
                                             prefix=f"te_{'_'.join(exist)}__{ACTION_COL}")
    X_test_te_list.append(te_df)

X_test_te_point_list = []
for feats in fe_sets:
    exist = [c for c in feats if c in train_df.columns and c in test_sel.columns]
    if not exist: 
        continue
    te_df, te_names = map_cond_probs_to_test(train_df, test_sel, exist, POINT_COL, point_classes, point_prior,
                                             prefix=f"te_{'_'.join(exist)}__{POINT_COL}")
    X_test_te_point_list.append(te_df)

X_test_action = pd.concat([X_test_base] + X_test_te_list, axis=1).fillna(0)
X_test_point  = pd.concat([X_test_base] + X_test_te_point_list, axis=1).fillna(0)

# ---------- 14) 先用 Task1 機率堆疊到 Task2 ----------
proba_action_train = model_action_full.predict_proba(X_all_te)
proba_cols = [f"act_proba_{i}" for i in range(proba_action_train.shape[1])]
proba_df   = pd.DataFrame(proba_action_train, columns=proba_cols, index=X_all_te.index)

X_point_plus = pd.concat([X_all_te_point, proba_df], axis=1)

# 重新訓練 Task2（用 action 機率特徵）
model_point_full  = LGBMClassifier(**lgb_params_mc)
model_point_full.fit(X_point_plus, y_point)

# 測試集同樣加入 action 機率特徵
proba_action_test = model_action_full.predict_proba(X_test_action)
proba_test_df     = pd.DataFrame(proba_action_test, columns=proba_cols, index=X_test_action.index)
X_test_point_plus = pd.concat([X_test_point, proba_test_df], axis=1)

# ---------- 15) 預測 ----------
pred_action = model_action_full.predict(X_test_action)               # Task1
pred_point  = model_point_full.predict(X_test_point_plus)            # Task2

if model_rally_full is not None:
    # 任務3避免使用 prevK_*（已在 feats_rally 控制）
    feats_rally_test = [f for f in X_test_base.columns if f in train_df.columns and f in feats_rally]
    X_test_rally = test_sel[feats_rally_test].fillna(0) if len(feats_rally_test)>0 else np.zeros((len(test_sel),1))
    prob_rally   = model_rally_full.predict_proba(X_test_rally)[:, 1]
else:
    prob_rally   = np.full(len(test_sel), 0.5)

# ---------- 16) 組裝 submission（照 sample 欄位覆寫） ----------
sub = sample_sub.copy()
sub["serverGetPoint"] = prob_rally
sub["pointId"]        = pred_point.astype(int)
sub["actionId"]       = pred_action.astype(int)

assert len(sub) == len(sample_sub) == len(test_sel)
assert (sub[KEY] == sample_sub[KEY]).all(), "❌ rally_uid 順序不一致"

sub.to_csv("submission.csv", index=False)
print("🎉 已輸出 submission.csv，列數 =", len(sub))


clean_train: (97962, 32)
clean_test : (3084, 29)
sample_sub : (1001, 4)
✅ 標籤：next_actionId, next_pointId, 任務3：serverGetPoint
實際使用特徵數： 15
CV Task1 (action) ...
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.045104 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 7934
[LightGBM] [Info] Number of data points in the train set: 78310, number of used features: 55
[LightGBM] [Info] Start training from score -2.995732
[LightGBM] [Info] Start training from score -2.995732
[LightGBM] [Info] Start training from score -2.995732
[LightGBM] [Info] Start training from score -2.995732
[LightGBM] [Info] Start training from score -2.995732
[LightGBM] [Info] Start training from score -2.995732
[LightGBM] [Info] Start training from score -2.995732
[LightGBM] [Info] Start training from score -2.995732
[LightGBM] [Info] Start training from score -2.995732
[LightGBM] [Info] Start training from score -2.995732
[LightGBM] [

Exception ignored on calling ctypes callback function <function _log_callback at 0x0000025AF2C28F40>:
Traceback (most recent call last):
  File "c:\Users\winni\AppData\Local\Programs\Python\Python313\Lib\site-packages\lightgbm\basic.py", line 287, in _log_callback
    def _log_callback(msg: bytes) -> None:
KeyboardInterrupt: 


No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with po

Exception ignored on calling ctypes callback function <function _log_callback at 0x0000025AF2C28F40>:
Traceback (most recent call last):
  File "c:\Users\winni\AppData\Local\Programs\Python\Python313\Lib\site-packages\lightgbm\basic.py", line 287, in _log_callback
    def _log_callback(msg: bytes) -> None:
KeyboardInterrupt: 


No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with po